1.Retrive the employees who earn more than thier manager

In [0]:
spark.sql(""" select * from my_learning.my_raw_feed.employees""").show(truncate=False)

In [0]:
from pyspark.sql import Row

# Create sample data for employees table
sample_employees = [
    Row(EMPLOYEE_ID='1', SALARY=1000, MANAGER_ID=None),
    Row(EMPLOYEE_ID='2', SALARY=900, MANAGER_ID='1'),
    Row(EMPLOYEE_ID='3', SALARY=800, MANAGER_ID='1'),
    Row(EMPLOYEE_ID='4', SALARY=950, MANAGER_ID='2'),
    Row(EMPLOYEE_ID='5', SALARY=1100, MANAGER_ID=None)
]

df_employees = spark.createDataFrame(sample_employees)
df_employees.createOrReplaceTempView("employees")

#mtd1
display(spark.sql("""
  SELECT e.EMPLOYEE_ID, e.SALARY, m.EMPLOYEE_ID AS MANAGER_ID, m.SALARY AS MANAGER_SALARY
  FROM employees e
  JOIN employees m
    ON cast(coalesce(e.MANAGER_ID,'' ) as STRING) = cast(m.EMPLOYEE_ID as STRING)
  WHERE e.SALARY > m.SALARY
"""))



In [0]:
spark.sql(""" select * from my_learning.my_raw_feed.employees""").printSchema()

In [0]:
from pyspark.sql import Row

# Create sample data for employees table
sample_employees = [
    Row(EMPLOYEE_ID='1', SALARY=1000, MANAGER_ID=None),
    Row(EMPLOYEE_ID='2', SALARY=900, MANAGER_ID='1'),
    Row(EMPLOYEE_ID='3', SALARY=800, MANAGER_ID='1'),
    Row(EMPLOYEE_ID='4', SALARY=950, MANAGER_ID='2'),
    Row(EMPLOYEE_ID='5', SALARY=1100, MANAGER_ID=None)
]

df_employees = spark.createDataFrame(sample_employees)
df_manager = spark.createDataFrame(sample_employees)

# Join employees with their managers and filter where employee salary > manager salary
df = df_employees.join(
    df_manager,
    df_employees.MANAGER_ID == df_manager.EMPLOYEE_ID,
    "inner"
).filter(
    df_employees.SALARY > df_manager.SALARY
).select(
    df_employees.EMPLOYEE_ID.alias("employee_id"),
    df_manager.EMPLOYEE_ID.alias("manager_id"),
    df_manager.SALARY.alias("manager_salary"),
    df_employees.SALARY.alias("employee_salary")
)

display(df)

2.count employees in each department having more than 5 employees

In [0]:
spark.sql(""" select * from my_learning.my_raw_feed.employees""").show(truncate=False)

In [0]:
spark.sql(""" select DEPARTMENT_ID,count(*) as cnt from my_learning.my_raw_feed.employees group by DEPARTMENT_ID having count(*) >5 """).show(truncate=False)


In [0]:
spark.sql(""" select DEPARTMENT_ID,count(*) as cnt from my_learning.my_raw_feed.employees group by DEPARTMENT_ID order by cnt desc """).show(truncate=False)


Find emplyees who join in last 6 months

In [0]:
spark.sql(""" select EMPLOYEE_ID,date_format(to_date(HIRE_DATE, 'dd-MMM-yy'), 'yyyy-MM-dd') as hire_date1 from my_learning.my_raw_feed.employees group by EMPLOYEE_ID,hire_date1 order by hire_date1""").show(truncate=False)



In [0]:
spark.sql(""" select EMPLOYEE_ID,date_format(to_date(HIRE_DATE, 'yy-MMM-dd'), 'yyyy-MM-dd') as hire_date1 from my_learning.my_raw_feed.employees where date_format(to_date(HIRE_DATE, 'yy-MMM-dd'), 'yyyy-MM-dd') > '2026-01-01' """).show(truncate=False)

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, lag, sum as spark_sum
from pyspark.sql.window import Window

# Sample login data: USER_ID, LOGIN_DATE
sample_logins = [
    Row(USER_ID='A', LOGIN_DATE='2026-07-15'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-16'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-17'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-18'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-20'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-15'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-16'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-17'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-18'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-19'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-20'),
    Row(USER_ID='C', LOGIN_DATE='2026-07-15'),
    Row(USER_ID='C', LOGIN_DATE='2026-07-17'),
    Row(USER_ID='C', LOGIN_DATE='2026-07-18'),
]

df_logins = spark.createDataFrame(sample_logins)

# Window for each user ordered by login date
w = Window.partitionBy("USER_ID").orderBy("LOGIN_DATE")

# Calculate difference in days between consecutive logins
df_logins = df_logins.withColumn(
    "prev_date",
    lag("LOGIN_DATE").over(w)
).withColumn(
    "diff",
    (col("LOGIN_DATE").cast("date").cast("long") - col("prev_date").cast("date").cast("long"))
)

# Identify consecutive login groups
df_logins = df_logins.withColumn(
    "group",
    spark_sum((col("diff") != 1).cast("int")).over(w)
)

# Find max streak per user
streaks = df_logins.groupBy("USER_ID", "group").count()
max_streaks = streaks.groupBy("USER_ID").agg({"count": "max"}).withColumnRenamed("max(count)", "max_streak")

# Find user(s) with the longest streak
longest_streak = max_streaks.agg({"max_streak": "max"}).collect()[0][0]
result = max_streaks.filter(col("max_streak") == longest_streak)

display(result)

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, lag, sum as spark_sum
from pyspark.sql.window import Window

# Sample login data: USER_ID, LOGIN_DATE
sample_logins = [
    Row(USER_ID='A', LOGIN_DATE='2026-07-15'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-16'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-17'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-18'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-20'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-15'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-16'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-17'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-18'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-19'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-20'),
    Row(USER_ID='C', LOGIN_DATE='2026-07-15'),
    Row(USER_ID='C', LOGIN_DATE='2026-07-17'),
    Row(USER_ID='C', LOGIN_DATE='2026-07-18'),
]

df_logins = spark.createDataFrame(sample_logins)


df_logins.createOrReplaceTempView("logins")
spark.sql(""" select USER_ID,count(*) as cnt from logins group by USER_ID """)

In [0]:
spark.sql(""" select * from Logins""").show(truncate=False)

In [0]:
spark.sql(""" select user_id ,login_date, lag(login_date,1) over(partition by user_id order by login_date) as prev_date from Logins""").show(truncate=False)

In [0]:
spark.sql(""" 
          
          with tbl1 as (
              select user_id ,login_date, (to_date(login_date,'yyyy-MM-dd') - dense_rank() over(partition by user_id order by user_id,login_date)) as grp_date from Logins
              )

            select user_id,login_date,grp_date from tbl1
              
              
              """).show(truncate=False)

In [0]:
spark.sql(""" 
          
          with tbl1 as (
              select user_id ,login_date, (to_date(login_date,'yyyy-MM-dd') - dense_rank() over(partition by user_id order by user_id,login_date)) as grp_date from Logins
              )

            select user_id,grp_date,count(*) from tbl1 group by user_id,grp_date order by grp_date
              
              
              """).show(truncate=False)

In [0]:
spark.sql(""" 
          
          with tbl1 as (
              select user_id ,login_date, (to_date(login_date,'yyyy-MM-dd') - dense_rank() over(partition by user_id order by user_id,login_date)) as grp_date from Logins
              )

            select user_id,grp_date,count(*) from tbl1 group by user_id,grp_date having count(*) > 5;
              
              
              """).show(truncate=False)

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, lag, sum as spark_sum
from pyspark.sql.window import Window

# Sample login data: USER_ID, LOGIN_DATE
sample_logins = [
    Row(USER_ID='A', LOGIN_DATE='2026-07-15'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-16'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-17'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-18'),
    Row(USER_ID='A', LOGIN_DATE='2026-07-20'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-15'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-16'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-17'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-18'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-19'),
    Row(USER_ID='B', LOGIN_DATE='2026-07-20'),
    Row(USER_ID='C', LOGIN_DATE='2026-07-15'),
    Row(USER_ID='C', LOGIN_DATE='2026-07-17'),
    Row(USER_ID='C', LOGIN_DATE='2026-07-18'),
]

df_logins = spark.createDataFrame(sample_logins)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df=df_logins.withColumn('grp_date',dense_rank().over(Window.partitionBy('user_id').orderBy('user_id','login_date'))).show(truncate=False)

In [0]:
df=df_logins \
    .withColumn('grp_rnk', dense_rank().over(Window.partitionBy('user_id').orderBy('user_id','login_date')))\
    .withColumn('grp_date',(to_date(col('login_date'),'yyyy-MM-dd')- col('grp_rnk'))).show(truncate=False)

In [0]:
df=df_logins \
    .withColumn('grp_rnk', dense_rank().over(Window.partitionBy('user_id').orderBy('user_id','login_date')))\
    .withColumn('grp_date',(to_date(col('login_date'),'yyyy-MM-dd')- col('grp_rnk'))).agg(count(col('grp_date'))).alias("cnt").selectExpr('user_id','cnt','grp_date').show(truncate=False)

In [0]:
from pyspark.sql.functions import to_date, col, dense_rank, count
from pyspark.sql.window import Window

w = Window.partitionBy('USER_ID').orderBy('USER_ID', 'LOGIN_DATE')

df_grp = df_logins \
    .withColumn('grp_rnk', dense_rank().over(w)) \
    .withColumn('grp_date', to_date(col('LOGIN_DATE'), 'yyyy-MM-dd') - col('grp_rnk'))

df_result = df_grp.groupBy('USER_ID', 'grp_date').agg(count('*').alias('cnt')).filter(col('cnt') > 5)

display(df_result)

Month over month sales growth 


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, sum as spark_sum, lag, round
from pyspark.sql.window import Window

# Sample sales data: PRODUCT_ID, SALE_DATE, SALES_AMOUNT
sample_sales = [
    Row(PRODUCT_ID='P1', SALE_DATE='2026-01-01', SALES_AMOUNT=100),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-02-01', SALES_AMOUNT=120),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-03-01', SALES_AMOUNT=130),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-04-01', SALES_AMOUNT=140),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-05-01', SALES_AMOUNT=150),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-06-01', SALES_AMOUNT=160),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-07-01', SALES_AMOUNT=170),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-08-01', SALES_AMOUNT=180),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-09-01', SALES_AMOUNT=190),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-10-01', SALES_AMOUNT=200),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-11-01', SALES_AMOUNT=210),
    Row(PRODUCT_ID='P1', SALE_DATE='2026-12-01', SALES_AMOUNT=220),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-01-01', SALES_AMOUNT=80),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-02-01', SALES_AMOUNT=90),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-03-01', SALES_AMOUNT=100),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-04-01', SALES_AMOUNT=110),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-05-01', SALES_AMOUNT=120),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-06-01', SALES_AMOUNT=130),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-07-01', SALES_AMOUNT=140),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-08-01', SALES_AMOUNT=150),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-09-01', SALES_AMOUNT=160),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-10-01', SALES_AMOUNT=170),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-11-01', SALES_AMOUNT=180),
    Row(PRODUCT_ID='P2', SALE_DATE='2026-12-01', SALES_AMOUNT=190),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-01-01', SALES_AMOUNT=60),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-02-01', SALES_AMOUNT=70),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-03-01', SALES_AMOUNT=80),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-04-01', SALES_AMOUNT=90),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-05-01', SALES_AMOUNT=100),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-06-01', SALES_AMOUNT=110),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-07-01', SALES_AMOUNT=120),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-08-01', SALES_AMOUNT=130),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-09-01', SALES_AMOUNT=140),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-10-01', SALES_AMOUNT=150),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-11-01', SALES_AMOUNT=160),
    Row(PRODUCT_ID='P3', SALE_DATE='2026-12-01', SALES_AMOUNT=170),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-01-01', SALES_AMOUNT=50),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-02-01', SALES_AMOUNT=55),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-03-01', SALES_AMOUNT=60),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-04-01', SALES_AMOUNT=65),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-05-01', SALES_AMOUNT=70),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-06-01', SALES_AMOUNT=75),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-07-01', SALES_AMOUNT=80),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-08-01', SALES_AMOUNT=85),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-09-01', SALES_AMOUNT=90),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-10-01', SALES_AMOUNT=95),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-11-01', SALES_AMOUNT=100),
    Row(PRODUCT_ID='P4', SALE_DATE='2026-12-01', SALES_AMOUNT=105),
]

df_sales = spark.createDataFrame(sample_sales)
df_sales.createOrReplaceTempView('sales')
display(df_sales)

In [0]:
spark.sql(""" select product_id,sale_date,sales_amount,lag(sale_date,1) over(partition by product_id order by sale_date) as prev_month, lag(sales_amount,1) over(partition by product_id order by sale_date) as prev_sales_amount from sales """).show(truncate=False)

In [0]:
spark.sql("""with tbl1 as ( select product_id,
          sale_date,
          sales_amount,
          lag(sale_date,1) over(partition by product_id order by sale_date) as prev_month, 
          lag(sales_amount,1) over(partition by product_id order by sale_date) as prev_sales_amount 
          from sales
          )

          select product_id,sale_date,sales_amount,prev_sales_amount,month(sale_date) as month_1,year(sale_date) as year_1,((sales_amount - prev_sales_amount)/prev_sales_amount)*100 as growth from tbl1 

           """).show(truncate=False)

In [0]:
spark.sql("""
    WITH sales_with_prev AS (
        SELECT
            product_id,
            sale_date,
            sales_amount,
            LAG(sales_amount, 1) OVER (PARTITION BY product_id ORDER BY sale_date) AS prev_sales_amount
        FROM sales
    )
    SELECT
        product_id,
        sale_date,
        sales_amount,
        prev_sales_amount,
        ((sales_amount - prev_sales_amount) / prev_sales_amount) * 100 AS month_over_month_growth
    FROM sales_with_prev
""").show(truncate=False)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
ws=Window.partitionBy("product_id").orderBy("sale_date")
df_prev_sales=df_sales\
    .withColumn("prev_month",lag(col("sale_date"),1).over(ws))\
    .withColumn('prev_sales_amt',lag(col('sales_amount'),1).over(ws))\
    .show(truncate=False)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
ws=Window.partitionBy("product_id").orderBy("sale_date")
df_prev_sales=df_sales\
    .withColumn("prev_month",lag(col("sale_date"),1).over(ws))\
    .withColumn('prev_sales_amt',lag(col('sales_amount'),1).over(ws))

df_grwth=df_prev_sales\
    .withColumn("grwth",((col('sales_amount')-col('prev_sales_amt'))/col('prev_sales_amt'))*100)\
    .show(truncate=False)